In [3]:
import seaborn as sns
import requests
import json
import pandas as pd
from joblib import Parallel, delayed

In [23]:
df = pd.read_parquet("10k_replays.parquet")
df = df.sample(n=500, random_state=42)
df

,uploadtime,id,format,players,rating,formatid,log
31512,1728307229,gen9ou-2217762019,[Gen 9] OU,"[ISMORISTT, DA Scizor Enjoyer]",1787.0,[Gen 9] OU,|badge|p1|bronze|gen9ou|100-3\n|uhtml|medal-ms...
13958,1728375530,gen9vgc2024regh-2218375698,[Gen 9] VGC 2024 Reg H,"[WithoutDoubts, ruinols]",1125.0,[Gen 9] VGC 2024 Reg H,|j|☆WithoutDoubts\n|j|☆ruinols\n|t:|1728375035...
38524,1728276135,gen9ou-2217560101,[Gen 9] OU,"[Heatranator, Sodamario]",1428.0,[Gen 9] OU,|j|☆Heatranator\n|j|☆Sodamario\n|t:|1728275632...
37012,1728280725,gen9ou-2217586660,[Gen 9] OU,"[shanecornak, Isky31]",1227.0,[Gen 9] OU,|j|☆shanecornak\n|j|☆Isky31\n|t:|1728280303\n|...
20023,1728351205,smogtours-gen3ou-796307,[Gen 3] OU,"[TefoShark, Big_Squatto]",NaN,[Gen 3] OU,|inactive|Battle timer is ON: inactive players...
...,...,...,...,...,...,...,...
10647,1728391928,gen9vgc2024regh-2218492632,[Gen 9] VGC 2024 Reg H,"[janksy365, ChuuPasseios]",1333.0,[Gen 9] VGC 2024 Reg H,|j|☆janksy365\n|j|☆ChuuPasseios\n|t:|172839148...
13972,1728375529,gen9vgc2024regh-2218375208,[Gen 9] VGC 2024 Reg H,"[Niko highway, strye]",NaN,[Gen 9] VGC 2024 Reg H,|j|☆Niko highway\n|j|☆strye\n|t:|1728374956\n|...
12912,1728380786,gen9ou-2218409446,[Gen 9] OU,"[Bunny56, FeraligatorBo]",1311.0,[Gen 9] OU,|j|☆Bunny56\n|j|☆FeraligatorBo\n|t:|1728380445...
23444,1728332276,gen9vgc2024regh-2218045461,[Gen 9] VGC 2024 Reg H,"[Lemon_mcgee, foxincrocs]",1166.0,[Gen 9] VGC 2024 Reg H,|j|☆Lemon_mcgee\n|j|☆foxincrocs\n|t:|172833206...


In [121]:
import time
N = 500
test_df = df.sample(n=N, random_state=42)

def get_replay_log(row_id):
    start = time.time()
    url = "http://localhost:8080/api/replays/batch"
    resp = requests.post(url, data={"ids": row_id})
    data = json.loads(resp.content[1:])[0]
    end = time.time()
    return data['log'], start, end

nprocs = 10
runner = Parallel(n_jobs=nprocs, backend="threading")
func = delayed(get_replay_log)
results = runner(func(x) for x in test_df.id.values)
test_df['log'] = [r[0] for r in results]
test_df['start'] = [r[1] for r in results]
test_df['end'] = [r[2] for r in results]
test_df['duration'] = test_df.end - test_df.start
total_time = test_df.end.max() - test_df.start.min()
print(f"Wall time: {total_time:.2f}s")
rate = total_time / N * 1000
print(f"Processing rate: {rate:.2f} ms/row")
throughput = N / total_time
print(f"Throughput: {throughput:.2f} rows/s")
print("Request duration stats:")
print(test_df.duration.describe())
display(test_df)

Wall time: 2.04s
Processing rate: 4.09 ms/row
Throughput: 244.66 rows/s
Request duration stats:
count    500.000000
mean       0.037564
std        0.014902
min        0.016053
25%        0.024416
50%        0.033355
75%        0.050325
max        0.079099
Name: duration, dtype: float64


,uploadtime,id,format,players,rating,formatid,log,start,end,duration
2455,1728418731,gen9vgc2024regh-2218794561,[Gen 9] VGC 2024 Reg H,"[Letorspoke, Omgitsdannyk]",1196.0,[Gen 9] VGC 2024 Reg H,|j|☆Letorspoke\n|j|☆Omgitsdannyk\n|t:|17284183...,1.731195e+09,1.731195e+09,0.029767
17641,1728361119,gen9nationaldex-2218291340,[Gen 9] National Dex,"[Grimm Roi, KantoChampGP]",1201.0,[Gen 9] National Dex,|j|☆Grimm Roi\n|j|☆KantoChampGP\n|t:|172836056...,1.731195e+09,1.731195e+09,0.067926
13341,1728378436,gen9vgc2024regh-2218393018,[Gen 9] VGC 2024 Reg H,"[balzy, Bicentenario]",1275.0,[Gen 9] VGC 2024 Reg H,|j|☆balzy\n|j|☆Bicentenario\n|t:|1728377826\n|...,1.731195e+09,1.731195e+09,0.049242
18522,1728356821,gen9randombattlesharedpowerb12p6-2218262268,"[Gen 9] Random Battle (Shared Power, B12P6)","[incontri蒂安希, Ashestoash]",1094.0,"[Gen 9] Random Battle (Shared Power, B12P6)",|j|☆incontri蒂安希\n|j|☆Ashestoash\n|t:|172835647...,1.731195e+09,1.731195e+09,0.046627
24147,1728329050,gen9randombattle-2218007038,[Gen 9] Random Battle,"[gsdfgsdfgr, Dellager_]",NaN,[Gen 9] Random Battle,|j|☆gsdfgsdfgr\n|j|☆Dellager_\n|t:|1728328614\...,1.731195e+09,1.731195e+09,0.063324
...,...,...,...,...,...,...,...,...,...,...
36346,1728283781,gen9brokencup-2217603961,[Gen 9] Broken Cup,"[thepokepersonthing, GDO_ERROR]",NaN,[Gen 9] Broken Cup,|j|☆thepokepersonthing\n|j|☆GDO_ERROR\n|t:|172...,1.731195e+09,1.731195e+09,0.045387
31332,1728307782,gen9doublesou-2217763030,[Gen 9] Doubles OU,"[Infinite Gumption, Lumandra]",1143.0,[Gen 9] Doubles OU,|j|☆Infinite Gumption\n|j|☆Lumandra\n|t:|17283...,1.731195e+09,1.731195e+09,0.048575
20463,1728349527,gen9vgc2024regh-2218201246,[Gen 9] VGC 2024 Reg H,"[Beh2242, DaisyDives]",1217.0,[Gen 9] VGC 2024 Reg H,|j|☆Beh2242\n|j|☆DaisyDives\n|t:|1728349089\n|...,1.731195e+09,1.731195e+09,0.036109
17771,1728360704,gen9vgc2024regh-2218288273,[Gen 9] VGC 2024 Reg H,"[Fmmf, Crescent29]",1082.0,[Gen 9] VGC 2024 Reg H,|j|☆Fmmf\n|j|☆Crescent29\n|t:|1728360091\n|gam...,1.731195e+09,1.731195e+09,0.040700


In [122]:
import time
N = 500
test_df = df.sample(n=N, random_state=42)

def get_replay_logs(row_ids):
    start = time.time()
    row_ids = list(row_ids)
    url = "http://localhost:8080/api/replays/batch"
    resp = requests.post(url, json={"ids": row_ids})
    data = json.loads(resp.content[1:])
    end = time.time()
    return data, start, end

nprocs = 10
runner = Parallel(n_jobs=nprocs, backend="threading")
func = delayed(get_replay_logs)
chunk_size = 50
num_chunks = N // chunk_size
packed_results = runner(func(x) for x in [test_df.id.values[i*chunk_size:(i+1)*chunk_size] for i in range(num_chunks)])
results = []
for data, start, end in packed_results:
    for row in data:
        row = row.copy()
        row.update({"start": start, "end": end})
        results.append(row)
results_df = pd.DataFrame(data=results)
results_df = results_df.set_index('id')
del test_df['log']
test_df = test_df.join(results_df.loc[:, ["log", "start", "end"]], on="id", how="left", validate="1:1")
test_df['duration'] = test_df.end - test_df.start
total_time = test_df.end.max() - test_df.start.min()
print(f"Wall time: {total_time:.2f}s")
rate = total_time / N * 1000
print(f"Processing rate: {rate:.2f} ms/row")
throughput = N / total_time
print(f"Throughput: {throughput:.2f} rows/s")
print("Request duration stats:")
print(test_df.duration.describe())
display(test_df)

Wall time: 0.14s
Processing rate: 0.27 ms/row
Throughput: 3680.06 rows/s
Request duration stats:
count    500.000000
mean       0.096307
std        0.022633
min        0.057426
25%        0.084101
50%        0.102022
75%        0.110535
max        0.134835
Name: duration, dtype: float64


,uploadtime,id,format,players,rating,formatid,log,start,end,duration
2455,1728418731,gen9vgc2024regh-2218794561,[Gen 9] VGC 2024 Reg H,"[Letorspoke, Omgitsdannyk]",1196.0,[Gen 9] VGC 2024 Reg H,|j|☆Letorspoke\n|j|☆Omgitsdannyk\n|t:|17284183...,1.731195e+09,1.731195e+09,0.101432
17641,1728361119,gen9nationaldex-2218291340,[Gen 9] National Dex,"[Grimm Roi, KantoChampGP]",1201.0,[Gen 9] National Dex,|j|☆Grimm Roi\n|j|☆KantoChampGP\n|t:|172836056...,1.731195e+09,1.731195e+09,0.101432
13341,1728378436,gen9vgc2024regh-2218393018,[Gen 9] VGC 2024 Reg H,"[balzy, Bicentenario]",1275.0,[Gen 9] VGC 2024 Reg H,|j|☆balzy\n|j|☆Bicentenario\n|t:|1728377826\n|...,1.731195e+09,1.731195e+09,0.101432
18522,1728356821,gen9randombattlesharedpowerb12p6-2218262268,"[Gen 9] Random Battle (Shared Power, B12P6)","[incontri蒂安希, Ashestoash]",1094.0,"[Gen 9] Random Battle (Shared Power, B12P6)",|j|☆incontri蒂安希\n|j|☆Ashestoash\n|t:|172835647...,1.731195e+09,1.731195e+09,0.101432
24147,1728329050,gen9randombattle-2218007038,[Gen 9] Random Battle,"[gsdfgsdfgr, Dellager_]",NaN,[Gen 9] Random Battle,|j|☆gsdfgsdfgr\n|j|☆Dellager_\n|t:|1728328614\...,1.731195e+09,1.731195e+09,0.101432
...,...,...,...,...,...,...,...,...,...,...
36346,1728283781,gen9brokencup-2217603961,[Gen 9] Broken Cup,"[thepokepersonthing, GDO_ERROR]",NaN,[Gen 9] Broken Cup,|j|☆thepokepersonthing\n|j|☆GDO_ERROR\n|t:|172...,1.731195e+09,1.731195e+09,0.110535
31332,1728307782,gen9doublesou-2217763030,[Gen 9] Doubles OU,"[Infinite Gumption, Lumandra]",1143.0,[Gen 9] Doubles OU,|j|☆Infinite Gumption\n|j|☆Lumandra\n|t:|17283...,1.731195e+09,1.731195e+09,0.110535
20463,1728349527,gen9vgc2024regh-2218201246,[Gen 9] VGC 2024 Reg H,"[Beh2242, DaisyDives]",1217.0,[Gen 9] VGC 2024 Reg H,|j|☆Beh2242\n|j|☆DaisyDives\n|t:|1728349089\n|...,1.731195e+09,1.731195e+09,0.110535
17771,1728360704,gen9vgc2024regh-2218288273,[Gen 9] VGC 2024 Reg H,"[Fmmf, Crescent29]",1082.0,[Gen 9] VGC 2024 Reg H,|j|☆Fmmf\n|j|☆Crescent29\n|t:|1728360091\n|gam...,1.731195e+09,1.731195e+09,0.110535
